# Petals to the Metal: corrida en Kaggle

Este notebook clona el repositorio del equipo y ejecuta el pipeline completo con `configs/kaggle.yaml` (resolución 224).

**Antes de correr:**
1. Agrega el dataset de la competencia *tpu-getting-started* (Add data → Competitions).
2. En *Settings*, activa **Internet** y elige el acelerador **GPU T4 x2 / P100**. La ruta de TPU no está probada.

In [ ]:
!git clone -q -b feature/flower-cnn-dm https://github.com/Lunaaaalj/NeuralNetworksTeamEnvironment.git
%cd NeuralNetworksTeamEnvironment
import tensorflow as tf
print(tf.__version__, tf.config.list_physical_devices())

## 1. Pruebas unitarias (datos sintéticos, ~30 s)

In [ ]:
!python -m pytest -q

## 2. Caché por clase a 224 px (~1.9 GB en disco)

In [ ]:
!python -m src.data --config configs/kaggle.yaml --build-cache

## 3. Experimentos

Destila los IPC que falten y entrena **completo / aleatorio / destilado** con 3 semillas. Los resultados se agregan a `reports/resultados.csv`.

Para una corrida corta, edita `experiments` en `configs/kaggle.yaml` (por ejemplo `ipcs: [10]`, `seeds: [0]`).

In [ ]:
!python -m scripts.run_experiments --config configs/kaggle.yaml

In [ ]:
import pandas as pd
from IPython.display import Image, display
df = pd.read_csv("reports/resultados.csv")
display(df.groupby(["data", "ipc"], dropna=False)[["val_f1_macro", "val_accuracy", "train_seconds", "distill_seconds"]].agg(["mean", "std"]).round(4))
display(Image("reports/figures/synthetic_ipc10_res224.png"))

## 4. Submission con el mejor modelo según F1 macro en val

In [ ]:
best = df.sort_values("val_f1_macro", ascending=False).iloc[0]
print(best[["run", "val_f1_macro"]].to_dict())
!python -m src.evaluate --config configs/kaggle.yaml --model {best.model_path} --make-submission --submission-path /kaggle/working/submission.csv